In [32]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings

In [33]:
# =============================================================================
# CELL 1: Configuration and Setup (Climate Data)
# =============================================================================

NOTEBOOK_NAME = "03: Climate Data Ingestion"
DATA_SUBFOLDER = "climate"
OBJECTIVE = """Process EUROSTAT climate data (2012-2024)
Heating/Cooling Degree Days for Austria and EU27_2020
Output: Monthly aggregated data only

IMPORTANT ARCHITECTURAL DECISION:
Climate data is MONTHLY ONLY - no daily or weekly rows will be created.
This prevents adding ~5190 empty rows to data_consolidated.csv.
Climate variables can only be used in monthly-level analyses."""

# Climate-specific configuration
CLIMATE_FILE_NAME = "nrg_chddr2_m__custom_17979334_linear_2_0.csv"
DATE_COLUMN = "TIME_PERIOD"
DELIMITER = ","
DECIMAL_SEPARATOR = "."

# Geographic entities to process
GEO_AUSTRIA = "AT"
GEO_EU = "EU27_2020"

# Setup functions (reused from electricity/carbon - consistent)
def setup_pandas_options():
    """Configure pandas display options for better data inspection."""
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_rows', 100)  
    pd.set_option('display.width', None)
    warnings.filterwarnings('ignore')

def setup_project_paths(data_subfolder):
    """Set up project directory paths for any data source."""
    PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
    RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / data_subfolder
    PROCESSED_DATA_PATH = PROJECT_ROOT / 'data' / 'processed'
    PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)
    return PROJECT_ROOT, RAW_DATA_PATH, PROCESSED_DATA_PATH

def print_notebook_header(notebook_name, objective, raw_path, processed_path):
    """Print standardized notebook start information."""
    print("="*60)
    print(f"NOTEBOOK: {notebook_name}")
    print("="*60)
    print(f"Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Raw Data Path: {raw_path}")
    print(f"Processed Data Path: {processed_path}")
    print(f"Objective: {objective}")
    print("="*60)

# Setup execution
setup_pandas_options()
PROJECT_ROOT, RAW_DATA_PATH, PROCESSED_DATA_PATH = setup_project_paths(DATA_SUBFOLDER)
print_notebook_header(NOTEBOOK_NAME, OBJECTIVE, RAW_DATA_PATH, PROCESSED_DATA_PATH)

print("Cell 1: Configuration and Setup - READY")

NOTEBOOK: 03: Climate Data Ingestion
Start Time: 2025-09-29 13:57:13
Raw Data Path: c:\Users\paulr\Desktop\Capstone\data\raw\climate
Processed Data Path: c:\Users\paulr\Desktop\Capstone\data\processed
Objective: Process EUROSTAT climate data (2012-2024)
Heating/Cooling Degree Days for Austria and EU27_2020
Output: Monthly aggregated data only

IMPORTANT ARCHITECTURAL DECISION:
Climate data is MONTHLY ONLY - no daily or weekly rows will be created.
This prevents adding ~5190 empty rows to data_consolidated.csv.
Climate variables can only be used in monthly-level analyses.
Cell 1: Configuration and Setup - READY


In [34]:
# =============================================================================
# CELL 2: Missing Values Standardization Function (with Quality Control)
# =============================================================================

def standardize_missing_values(df, additional_missing=None, show_quality_control=True):
    """
    Convert various missing value representations to pandas NaN.
    
    Args:
        df (DataFrame): Input dataframe
        additional_missing (list): Extra missing indicators beyond defaults
        show_quality_control (bool): Whether to display quality control output
    
    Returns:
        DataFrame: Dataframe with standardized missing values
        dict: Report of found missing patterns
    """
    missing_indicators = [
        'N/A', 'n/a', 'NA', 'na',
        '', '-', '--', '---',
        'NULL', 'null', 'Null',
        'NaN', 'nan', '#N/A'
    ]
    
    if additional_missing:
        missing_indicators.extend(additional_missing)
    
    if show_quality_control:
        print("\nMISSING VALUES STANDARDIZATION - QUALITY CONTROL:")
        print("-" * 55)
        print("Missing indicators standardized:")
        print("  ", ', '.join([f"'{ind}'" for ind in missing_indicators]))
        print()
    
    found_patterns = {}
    df_clean = df.copy()
    
    for col in df_clean.columns:
        original_nulls = df_clean[col].isnull().sum()
        df_clean[col] = df_clean[col].replace(missing_indicators, np.nan)
        new_nulls = df_clean[col].isnull().sum()
        converted_count = new_nulls - original_nulls
        
        if converted_count > 0:
            found_patterns[col] = {
                'original_nulls': original_nulls,
                'converted_missing': converted_count,
                'total_nulls': new_nulls
            }
    
    if show_quality_control:
        if found_patterns:
            print("CONVERSION RESULTS BY COLUMN:")
            for col, pattern in found_patterns.items():
                print(f"  {col}:")
                print(f"    Original nulls: {pattern['original_nulls']}")
                print(f"    Converted missing: {pattern['converted_missing']}")
                print(f"    Total nulls: {pattern['total_nulls']}")
        else:
            print("No missing value patterns found for conversion")
        print("-" * 55)
    
    return df_clean, found_patterns

print("Cell 2: Missing Values Standardization Function - READY")

Cell 2: Missing Values Standardization Function - READY


In [35]:
# =============================================================================
# CELL 3: Climate File Discovery
# =============================================================================

def discover_climate_file(data_path, filename):
    """
    Discover single climate data file.
    
    Args:
        data_path (Path): Path to raw data directory
        filename (str): Expected filename
    
    Returns:
        dict: File information dictionary
    """
    file_path = data_path / filename
    
    if file_path.exists():
        return {
            'filename': filename,
            'path': str(file_path),
            'exists': True
        }
    else:
        return {
            'filename': filename,
            'path': str(file_path),
            'exists': False
        }

def display_file_discovery_results(file_info):
    """Display results of climate file discovery."""
    print("Climate file discovery:")
    print(f"  File: {file_info['filename']}")
    print(f"  Exists: {file_info['exists']}")
    print(f"  Path: {file_info['path']}")
    
    if file_info['exists']:
        print("Ready to proceed with climate data loading")
    else:
        print("ERROR: Climate file not found!")

# Execute file discovery
climate_file_info = discover_climate_file(RAW_DATA_PATH, CLIMATE_FILE_NAME)
display_file_discovery_results(climate_file_info)

print("Cell 3: Climate File Discovery - COMPLETE")

Climate file discovery:
  File: nrg_chddr2_m__custom_17979334_linear_2_0.csv
  Exists: True
  Path: c:\Users\paulr\Desktop\Capstone\data\raw\climate\nrg_chddr2_m__custom_17979334_linear_2_0.csv
Ready to proceed with climate data loading
Cell 3: Climate File Discovery - COMPLETE


In [36]:
# =============================================================================
# CELL 4: Load and Clean Climate Data
# =============================================================================

def load_climate_data(file_path, date_column='TIME_PERIOD'):
    """
    Load climate data and filter to relevant columns only.
    
    Args:
        file_path (str/Path): Path to climate CSV file
        date_column (str): Name of date column
    
    Returns:
        tuple: (dataframe, metadata)
    """
    try:
        # Load complete dataset
        df = pd.read_csv(file_path, delimiter=DELIMITER, decimal=DECIMAL_SEPARATOR)
        
        print(f"INITIAL DATA LOAD:")
        print(f"  Original shape: {df.shape}")
        print(f"  All columns: {df.columns.tolist()}")
        print()
        
        # Filter to only relevant columns (4 of 19)
        relevant_columns = [date_column, 'indic_nrg', 'geo', 'OBS_VALUE']
        df_filtered = df[relevant_columns].copy()
        
        print(f"COLUMN FILTERING:")
        print(f"  Kept {len(relevant_columns)} of {len(df.columns)} columns")
        print(f"  Relevant columns: {relevant_columns}")
        print()
        
        # Clean missing values
        df_clean, missing_patterns = standardize_missing_values(df_filtered, show_quality_control=True)
        
        # Convert date column to datetime (YYYY-MM format)
        df_clean[date_column] = pd.to_datetime(df_clean[date_column], format='%Y-%m')
        
        # Create metadata (consistent with electricity/carbon structure)
        metadata = {
            'filename': Path(file_path).name,
            'rows': len(df_clean),
            'columns': list(df_clean.columns),
            'geographic_entities': df_clean['geo'].unique().tolist(),
            'indicators': df_clean['indic_nrg'].unique().tolist(),
            'missing_patterns_found': missing_patterns,
            'date_range': (df_clean[date_column].min(), df_clean[date_column].max()) if len(df_clean) > 0 else (None, None)
        }
        
        print(f"DATA SUMMARY:")
        print(f"  Final shape: {df_clean.shape}")
        print(f"  Date range: {metadata['date_range'][0].strftime('%Y-%m')} to {metadata['date_range'][1].strftime('%Y-%m')}")
        print(f"  Geographic entities: {metadata['geographic_entities']}")
        print(f"  Indicators: {metadata['indicators']}")
        print()
        
        print("DISTRIBUTION BY GEO AND INDICATOR:")
        print(df_clean.groupby(['geo', 'indic_nrg']).size())
        print()
        
        print("SAMPLE DATA:")
        print(df_clean.head(10))
        
        return df_clean, metadata
        
    except Exception as e:
        return None, {'error': str(e)}

# Execute data loading
if climate_file_info['exists']:
    climate_df, climate_metadata = load_climate_data(climate_file_info['path'])
    
    if climate_df is None:
        print(f"ERROR loading climate data: {climate_metadata['error']}")
else:
    print("Skipping data load - file not found")

print("Cell 4: Load and Clean Climate Data - COMPLETE")

INITIAL DATA LOAD:
  Original shape: (608, 19)
  All columns: ['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'freq', 'Time frequency', 'unit', 'Unit of measure', 'indic_nrg', 'Energy indicator', 'geo', 'Geopolitical entity (reporting)', 'TIME_PERIOD', 'Time', 'OBS_VALUE', 'Observation value', 'OBS_FLAG', 'Observation status (Flag) V2 structure', 'CONF_STATUS', 'Confidentiality status (flag)']

COLUMN FILTERING:
  Kept 4 of 19 columns
  Relevant columns: ['TIME_PERIOD', 'indic_nrg', 'geo', 'OBS_VALUE']


MISSING VALUES STANDARDIZATION - QUALITY CONTROL:
-------------------------------------------------------
Missing indicators standardized:
   'N/A', 'n/a', 'NA', 'na', '', '-', '--', '---', 'NULL', 'null', 'Null', 'NaN', 'nan', '#N/A'

No missing value patterns found for conversion
-------------------------------------------------------
DATA SUMMARY:
  Final shape: (608, 4)
  Date range: 2012-05 to 2024-12
  Geographic entities: ['AT', 'EU27_2020']
  Indicators: ['CDD', 'HDD']

DISTRIB

In [37]:
# =============================================================================
# CELL 5: Pivot Austria and EU Climate Data
# =============================================================================

def pivot_climate_by_geography(df, geo_code, date_column='TIME_PERIOD'):
    """
    Pivot climate data for a specific geography from long to wide format.
    Converts indic_nrg values (CDD, HDD) from rows to columns.
    
    Args:
        df (DataFrame): Climate data in long format
        geo_code (str): Geographic code to filter ('AT' or 'EU27_2020')
        date_column (str): Name of date column
    
    Returns:
        DataFrame: Pivoted climate data with CDD and HDD as columns
    """
    print(f"PIVOTING CLIMATE DATA FOR: {geo_code}")
    print("-" * 50)
    
    df_geo = df[df['geo'] == geo_code].copy()
    
    print(f"Before pivot: {len(df_geo)} rows")
    
    df_pivoted = df_geo.pivot(index=date_column, columns='indic_nrg', values='OBS_VALUE')
    df_pivoted.reset_index(inplace=True)
    df_pivoted.columns.name = None
    
    print(f"After pivot: {len(df_pivoted)} rows (reduction: {len(df_geo)/len(df_pivoted):.1f}x)")
    print(f"Columns: {df_pivoted.columns.tolist()}")
    print(f"Sample:\n{df_pivoted.head(3)}\n")
    
    return df_pivoted

# Execute pivots for both geographies
if 'climate_df' in locals() and climate_df is not None:
    climate_austria = pivot_climate_by_geography(climate_df, GEO_AUSTRIA)
    climate_eu = pivot_climate_by_geography(climate_df, GEO_EU)
    
    # Verify alignment
    print("TIME_PERIOD ALIGNMENT CHECK:")
    at_dates = set(climate_austria[DATE_COLUMN])
    eu_dates = set(climate_eu[DATE_COLUMN])
    print(f"  Match: {at_dates == eu_dates}")

print("Cell 5: Pivot Austria and EU Climate Data - COMPLETE")

PIVOTING CLIMATE DATA FOR: AT
--------------------------------------------------
Before pivot: 304 rows
After pivot: 152 rows (reduction: 2.0x)
Columns: ['TIME_PERIOD', 'CDD', 'HDD']
Sample:
  TIME_PERIOD    CDD     HDD
0  2012-05-01   0.05  143.18
1  2012-06-01   8.14   56.20
2  2012-07-01  13.45   36.62

PIVOTING CLIMATE DATA FOR: EU27_2020
--------------------------------------------------
Before pivot: 304 rows
After pivot: 152 rows (reduction: 2.0x)
Columns: ['TIME_PERIOD', 'CDD', 'HDD']
Sample:
  TIME_PERIOD    CDD     HDD
0  2012-05-01   1.93  134.05
1  2012-06-01  18.99   67.23
2  2012-07-01  41.82   25.92

TIME_PERIOD ALIGNMENT CHECK:
  Match: True
Cell 5: Pivot Austria and EU Climate Data - COMPLETE


In [38]:
# =============================================================================
# CELL 6: Merge Austria and EU Climate Data (Horizontal)
# =============================================================================

def merge_climate_geographies(df_austria, df_eu, date_column='TIME_PERIOD'):
    """
    Merge Austria and EU climate data horizontally (add EU columns to Austria).
    
    Args:
        df_austria (DataFrame): Pivoted Austria climate data
        df_eu (DataFrame): Pivoted EU climate data
        date_column (str): Name of date column for merge
    
    Returns:
        DataFrame: Merged climate data with both AT and EU columns
    """
    print("MERGING AUSTRIA AND EU CLIMATE DATA:")
    print("-" * 50)
    
    # Rename indicator columns to include geography suffix
    df_austria_renamed = df_austria.copy()
    df_austria_renamed.rename(columns={
        'CDD': 'CDD_AT',
        'HDD': 'HDD_AT'
    }, inplace=True)
    
    df_eu_renamed = df_eu.copy()
    df_eu_renamed.rename(columns={
        'CDD': 'CDD_EU',
        'HDD': 'HDD_EU'
    }, inplace=True)
    
    print(f"Renamed columns:")
    print(f"  Austria: {[col for col in df_austria_renamed.columns if col != date_column]}")
    print(f"  EU: {[col for col in df_eu_renamed.columns if col != date_column]}")
    print()
    
    # Merge on TIME_PERIOD
    df_merged = pd.merge(
        df_austria_renamed,
        df_eu_renamed,
        on=date_column,
        how='outer'  # Keep all dates from both
    )
    
    print(f"Merge results:")
    print(f"  Austria rows: {len(df_austria_renamed)}")
    print(f"  EU rows: {len(df_eu_renamed)}")
    print(f"  Merged rows: {len(df_merged)}")
    print(f"  Final columns: {df_merged.columns.tolist()}")
    print()
    
    print("Sample merged data:")
    print(df_merged.head())
    print()
    
    # Check for missing values after merge
    print("Missing values in merged data:")
    for col in df_merged.columns:
        if col != date_column:
            missing = df_merged[col].isnull().sum()
            if missing > 0:
                print(f"  {col}: {missing} missing ({missing/len(df_merged)*100:.1f}%)")
    
    if df_merged.isnull().sum().sum() == 0:
        print("  No missing values - perfect merge")
    
    return df_merged

# Execute merge
if 'climate_austria' in locals() and 'climate_eu' in locals():
    if climate_austria is not None and climate_eu is not None:
        climate_merged = merge_climate_geographies(climate_austria, climate_eu)
        
        print("\nMERGE SUCCESSFUL")
        print(f"  Final shape: {climate_merged.shape}")
        print(f"  Date range: {climate_merged[DATE_COLUMN].min().strftime('%Y-%m')} to {climate_merged[DATE_COLUMN].max().strftime('%Y-%m')}")
    else:
        print("ERROR: One or both pivot results are None")
else:
    print("Skipping merge - pivoted data not available")

print("Cell 6: Merge Austria and EU Climate Data - COMPLETE")

MERGING AUSTRIA AND EU CLIMATE DATA:
--------------------------------------------------
Renamed columns:
  Austria: ['CDD_AT', 'HDD_AT']
  EU: ['CDD_EU', 'HDD_EU']

Merge results:
  Austria rows: 152
  EU rows: 152
  Merged rows: 152
  Final columns: ['TIME_PERIOD', 'CDD_AT', 'HDD_AT', 'CDD_EU', 'HDD_EU']

Sample merged data:
  TIME_PERIOD  CDD_AT  HDD_AT  CDD_EU  HDD_EU
0  2012-05-01    0.05  143.18    1.93  134.05
1  2012-06-01    8.14   56.20   18.99   67.23
2  2012-07-01   13.45   36.62   41.82   25.92
3  2012-08-01    9.88   25.47   44.96   32.69
4  2012-09-01    0.06  116.91    6.59   98.60

Missing values in merged data:
  No missing values - perfect merge

MERGE SUCCESSFUL
  Final shape: (152, 5)
  Date range: 2012-05 to 2024-12
Cell 6: Merge Austria and EU Climate Data - COMPLETE


In [39]:
# =============================================================================
# CELL 7: Standardize Column Names and Create Long Format
# =============================================================================

def standardize_climate_column_names(df, date_column='TIME_PERIOD'):
    """
    Standardize climate column names with climate_ prefix.
    
    Args:
        df (DataFrame): Merged climate data
        date_column (str): Name of date column to preserve
    
    Returns:
        DataFrame: Data with standardized column names
    """
    df_renamed = df.copy()
    rename_map = {}
    
    for col in df.columns:
        if col == date_column:
            continue
        else:
            # Create standardized name: CDD_AT -> climate_cdd_at
            new_name = col.lower()
            new_name = f"climate_{new_name}"
            rename_map[col] = new_name
    
    df_renamed.rename(columns=rename_map, inplace=True)
    
    print("COLUMN NAME STANDARDIZATION:")
    print("-" * 40)
    for old_name, new_name in rename_map.items():
        print(f"  {old_name} → {new_name}")
    
    return df_renamed

def transform_climate_to_long_format(df, date_column='TIME_PERIOD'):
    """
    Transform climate data to long format.
    IMPORTANT: Only creates MONTHLY rows (no daily/weekly as per architecture decision).
    
    Args:
        df (DataFrame): Climate data with standardized columns
        date_column (str): Name of date column
    
    Returns:
        DataFrame: Climate data in long format (monthly only)
    """
    print("\nTRANSFORMING TO LONG FORMAT:")
    print("-" * 40)
    print("ARCHITECTURAL NOTE: Creating MONTHLY rows only")
    print("No daily or weekly rows will be created for climate data")
    print()
    
    df_long = df.copy()
    
    # Convert TIME_PERIOD to datetime if not already
    df_long[date_column] = pd.to_datetime(df_long[date_column])
    
    # Add time components
    df_long['year'] = df_long[date_column].dt.year
    df_long['month'] = df_long[date_column].dt.month
    df_long['quarter'] = df_long[date_column].dt.quarter
    df_long['week'] = df_long[date_column].dt.isocalendar().week
    df_long['month_name'] = df_long[date_column].dt.month_name()
    
    # Add aggregation level - MONTHLY ONLY
    df_long['aggregation_level'] = 'monthly'
    
    # Convert date to first of month in ISO format
    df_long['date'] = df_long[date_column].apply(lambda x: x.replace(day=1).strftime('%Y-%m-%d'))
    
    # Convert all climate columns to float64 (like exchange rates, not Int64)
    climate_columns = [col for col in df_long.columns if col.startswith('climate_')]
    
    print(f"Converting {len(climate_columns)} climate columns to float64:")
    for col in climate_columns:
        df_long[col] = df_long[col].astype('float64')
        print(f"  {col}: float64")
    
    # Select final columns in correct order
    base_columns = ['date', 'year', 'month', 'quarter', 'week', 'aggregation_level', 'month_name']
    final_columns = base_columns + climate_columns
    
    # Drop TIME_PERIOD (replaced by date)
    df_final = df_long[final_columns].copy()
    
    print(f"\nFINAL STRUCTURE:")
    print(f"  Rows: {len(df_final)} (monthly only)")
    print(f"  Columns: {df_final.columns.tolist()}")
    print(f"  Date range: {df_final['date'].min()} to {df_final['date'].max()}")
    print()
    
    print("Sample data:")
    print(df_final.head())
    
    return df_final

# Execute standardization and transformation
if 'climate_merged' in locals() and climate_merged is not None:
    climate_standardized = standardize_climate_column_names(climate_merged)
    
    print(f"\nStandardized columns: {climate_standardized.columns.tolist()}\n")
    
    climate_long_format = transform_climate_to_long_format(climate_standardized)
    
    print("\nLONG FORMAT TRANSFORMATION SUCCESSFUL")
    print(f"  Shape: {climate_long_format.shape}")
    print(f"  Data types:")
    for col in climate_long_format.columns:
        if col.startswith('climate_'):
            print(f"    {col}: {climate_long_format[col].dtype}")
else:
    print("Skipping transformation - merged data not available")

print("Cell 7: Standardize Column Names and Create Long Format - COMPLETE")

COLUMN NAME STANDARDIZATION:
----------------------------------------
  CDD_AT → climate_cdd_at
  HDD_AT → climate_hdd_at
  CDD_EU → climate_cdd_eu
  HDD_EU → climate_hdd_eu

Standardized columns: ['TIME_PERIOD', 'climate_cdd_at', 'climate_hdd_at', 'climate_cdd_eu', 'climate_hdd_eu']


TRANSFORMING TO LONG FORMAT:
----------------------------------------
ARCHITECTURAL NOTE: Creating MONTHLY rows only
No daily or weekly rows will be created for climate data

Converting 4 climate columns to float64:
  climate_cdd_at: float64
  climate_hdd_at: float64
  climate_cdd_eu: float64
  climate_hdd_eu: float64

FINAL STRUCTURE:
  Rows: 152 (monthly only)
  Columns: ['date', 'year', 'month', 'quarter', 'week', 'aggregation_level', 'month_name', 'climate_cdd_at', 'climate_hdd_at', 'climate_cdd_eu', 'climate_hdd_eu']
  Date range: 2012-05-01 to 2024-12-01

Sample data:
         date  year  month  quarter  week aggregation_level month_name  \
0  2012-05-01  2012      5        2    18           monthl

In [ ]:
# =============================================================================
# CELL 8: Save Climate Dataset
# =============================================================================

def save_climate_dataset(df, output_dir, filename="climate_consolidated.csv"):
    """
    Save climate dataset as separate validation sample.
    
    Args:
        df (DataFrame): Final climate dataset
        output_dir (Path): Directory for outputs
        filename (str): Output filename
    
    Returns:
        Path: Path to saved file
    """
    if df is None or len(df) == 0:
        print("No climate data to save!")
        return None
    
    output_path = output_dir / filename
    df.to_csv(output_path, index=False, na_rep='')
    
    print(f"CLIMATE DATASET SAVED:")
    print(f"  Path: {output_path}")
    print(f"  Rows: {len(df)}")
    print(f"  Columns: {len(df.columns)}")
    print(f"  Aggregation level: {df['aggregation_level'].unique()}")
    print(f"  Date range: {df['date'].min()} to {df['date'].max()}")
    
    return output_path

# Save climate dataset as validation sample
if 'climate_long_format' in locals() and climate_long_format is not None:
    climate_file_path = save_climate_dataset(climate_long_format, PROCESSED_DATA_PATH)
    
    if climate_file_path:
        print("\nCLIMATE FORK SUCCESSFUL")
        print("Ready for merge with data_consolidated.csv")
else:
    print("Skipping save - climate_long_format not available")

print("Cell 8: Save Climate Dataset - COMPLETE")

CLIMATE DATASET SAVED:
  Path: c:\Users\paulr\Desktop\Capstone\data\processed\climate_consolidated.csv
  Rows: 152
  Columns: 11
  Aggregation level: ['monthly']
  Date range: 2012-05-01 to 2024-12-01

CLIMATE FORK SUCCESSFUL
Ready for merge with data_consolidated.csv
Cell 8: Save Climate Dataset - COMPLETE


In [ ]:
# =============================================================================
# CELL 9: Merge Climate with Consolidated Data
# =============================================================================

def merge_climate_with_consolidated_data(climate_df, consolidated_file_path, output_dir):
    """
    Merge climate data with existing consolidated data (electricity + carbon).
    
    Args:
        climate_df (DataFrame): Climate dataset in long format (monthly only)
        consolidated_file_path (str/Path): Path to data_consolidated.csv
        output_dir (Path): Directory for final output
    
    Returns:
        DataFrame: Merged dataset with climate data added
    """
    consolidated_path = Path(consolidated_file_path)
    
    if not consolidated_path.exists():
        print(f"ERROR: Consolidated file not found at {consolidated_path}")
        return pd.DataFrame()
    
    print("MERGING CLIMATE WITH CONSOLIDATED DATA:")
    print("-" * 50)
    
    try:
        # Load existing consolidated data (electricity + carbon)
        consolidated_df = pd.read_csv(consolidated_path)
        
        print(f"Existing consolidated data loaded:")
        print(f"  Shape: {consolidated_df.shape}")
        print(f"  Aggregation levels: {consolidated_df['aggregation_level'].value_counts().to_dict()}")
        print()
        
        print(f"Climate data to merge:")
        print(f"  Shape: {climate_df.shape}")
        print(f"  Aggregation levels: {climate_df['aggregation_level'].value_counts().to_dict()}")
        print(f"  Date range: {climate_df['date'].min()} to {climate_df['date'].max()}")
        print()
        
        # Merge on date and aggregation_level
        # IMPORTANT: Only monthly rows in climate will find matches
        merged_df = pd.merge(
            consolidated_df, 
            climate_df, 
            on=['date', 'aggregation_level'], 
            how='outer',  # Keep all rows from both datasets
            suffixes=('', '_climate')
        )
        
        print(f"After merge:")
        print(f"  Shape: {merged_df.shape}")
        print()
        
        # Handle duplicate columns from merge
        duplicate_cols = ['year', 'month', 'quarter', 'week', 'month_name']
        for col in duplicate_cols:
            climate_col = f"{col}_climate"
            if climate_col in merged_df.columns:
                # Use existing values, fill missing with climate values
                merged_df[col] = merged_df[col].fillna(merged_df[climate_col])
                merged_df.drop(climate_col, axis=1, inplace=True)
                print(f"  Resolved duplicate: {col}")
        
        # Sort by date and aggregation level
        merged_df = merged_df.sort_values(['date', 'aggregation_level']).reset_index(drop=True)
        
        # Save merged dataset (overwrites existing data_consolidated.csv)
        final_path = output_dir / "data_consolidated.csv"
        merged_df.to_csv(final_path, index=False, na_rep='')
        
        print(f"\nFINAL CONSOLIDATED DATASET:")
        print(f"  Saved to: {final_path}")
        print(f"  Final shape: {merged_df.shape}")
        print(f"  Date range: {merged_df['date'].min()} to {merged_df['date'].max()}")
        print(f"  Aggregation levels: {merged_df['aggregation_level'].value_counts().to_dict()}")
        print()
        
        # Show which columns are climate-specific
        climate_columns = [col for col in merged_df.columns if col.startswith('climate_')]
        print(f"Climate columns added ({len(climate_columns)}):")
        for col in climate_columns:
            non_null = merged_df[col].notna().sum()
            print(f"  {col}: {non_null} non-null values")
        print()
        
        print("Sample of merged data (showing climate columns):")
        sample_cols = ['date', 'aggregation_level'] + climate_columns[:4]
        print(merged_df[sample_cols].head(10))
        
        return merged_df
        
    except Exception as e:
        print(f"ERROR during merge: {e}")
        return pd.DataFrame()

# Execute merge with consolidated data
if 'climate_long_format' in locals() and climate_long_format is not None:
    consolidated_file = PROCESSED_DATA_PATH / "data_consolidated.csv"
    final_merged_df = merge_climate_with_consolidated_data(
        climate_long_format, 
        consolidated_file, 
        PROCESSED_DATA_PATH
    )
    
    if len(final_merged_df) > 0:
        print("\nCLIMATE MERGE SUCCESSFUL")
        print(f"data_consolidated.csv now contains: Electricity + Carbon + Climate")
    else:
        print("\nMERGE FAILED")
else:
    print("Skipping merge - climate_long_format not available")

print("Cell 9: Merge Climate with Consolidated Data - COMPLETE")

MERGING CLIMATE WITH CONSOLIDATED DATA:
--------------------------------------------------
Existing consolidated data loaded:
  Shape: (4590, 15)
  Aggregation levels: {'daily': 3896, 'weekly': 566, 'monthly': 128}

Climate data to merge:
  Shape: (152, 11)
  Aggregation levels: {'monthly': 152}
  Date range: 2012-05-01 to 2024-12-01

After merge:
  Shape: (4622, 24)

  Resolved duplicate: year
  Resolved duplicate: month
  Resolved duplicate: quarter
  Resolved duplicate: week
  Resolved duplicate: month_name

FINAL CONSOLIDATED DATASET:
  Saved to: c:\Users\paulr\Desktop\Capstone\data\processed\data_consolidated.csv
  Final shape: (4622, 19)
  Date range: 2012-05-01 to 2025-09-01
  Aggregation levels: {'daily': 3896, 'weekly': 566, 'monthly': 160}

Climate columns added (4):
  climate_cdd_at: 152 non-null values
  climate_hdd_at: 152 non-null values
  climate_cdd_eu: 152 non-null values
  climate_hdd_eu: 152 non-null values

Sample of merged data (showing climate columns):
         d

In [43]:
# Diagnostic Analysis for Final Consolidated Dataset

consolidated_path = PROCESSED_DATA_PATH / "data_consolidated.csv"
df_final = pd.read_csv(consolidated_path)

print("="*70)
print("FINAL CONSOLIDATED DATASET DIAGNOSTICS")
print("="*70)

# 1. Basic Structure
print("\n1. BASIC STRUCTURE:")
print(f"   Shape: {df_final.shape}")
print(f"   Columns ({len(df_final.columns)}): {df_final.columns.tolist()}")

# 2. Aggregation Level Distribution
print("\n2. AGGREGATION LEVEL DISTRIBUTION:")
print(df_final['aggregation_level'].value_counts().sort_index())

# 3. Date Range by Aggregation Level
print("\n3. DATE RANGE BY AGGREGATION LEVEL:")
for level in ['daily', 'weekly', 'monthly']:
    level_data = df_final[df_final['aggregation_level'] == level]
    if len(level_data) > 0:
        print(f"   {level.capitalize()}: {level_data['date'].min()} to {level_data['date'].max()} ({len(level_data)} rows)")

# 4. Column Groups and Missing Values
print("\n4. MISSING VALUES BY DATA SOURCE:")

electricity_cols = [col for col in df_final.columns if 'price_exaa' in col or 'price_mc' in col or 'price_count' in col]
carbon_cols = [col for col in df_final.columns if col.startswith('carbonprices_')]
climate_cols = [col for col in df_final.columns if col.startswith('climate_')]

print(f"\n   Electricity columns ({len(electricity_cols)}):")
for col in electricity_cols:
    missing = df_final[col].isnull().sum()
    print(f"      {col}: {missing} missing ({missing/len(df_final)*100:.1f}%)")

print(f"\n   Carbon columns ({len(carbon_cols)}):")
for col in carbon_cols:
    missing = df_final[col].isnull().sum()
    print(f"      {col}: {missing} missing ({missing/len(df_final)*100:.1f}%)")

print(f"\n   Climate columns ({len(climate_cols)}):")
for col in climate_cols:
    missing = df_final[col].isnull().sum()
    print(f"      {col}: {missing} missing ({missing/len(df_final)*100:.1f}%)")

# 5. Data Types Verification
print("\n5. DATA TYPES VERIFICATION:")
print("   Electricity prices:", df_final['price_exaa_mean'].dtype if 'price_exaa_mean' in df_final.columns else 'N/A')
print("   Carbon prices:", df_final['carbonprices_primary_market'].dtype if 'carbonprices_primary_market' in df_final.columns else 'N/A')
print("   Carbon exchange rate:", df_final['carbonprices_exchange_rate_eur_usd'].dtype if 'carbonprices_exchange_rate_eur_usd' in df_final.columns else 'N/A')
print("   Climate degree days:", df_final['climate_cdd_at'].dtype if 'climate_cdd_at' in df_final.columns else 'N/A')

# 6. Sample Data Showing All Sources
print("\n6. SAMPLE DATA (Monthly rows showing all data sources):")
monthly_sample = df_final[df_final['aggregation_level'] == 'monthly'].head(5)
display_cols = ['date', 'aggregation_level', 'price_exaa_mean', 'carbonprices_primary_market', 'climate_cdd_at', 'climate_hdd_at']
existing_cols = [col for col in display_cols if col in monthly_sample.columns]
print(monthly_sample[existing_cols])

# 7. Climate Data Availability Check
print("\n7. CLIMATE DATA AVAILABILITY:")
print(f"   Total rows: {len(df_final)}")
print(f"   Rows with climate data: {df_final['climate_cdd_at'].notna().sum()}")
print(f"   Climate only in monthly: {(df_final['aggregation_level'] == 'monthly').sum()} monthly rows exist")
print(f"   Daily/Weekly with climate: {df_final[(df_final['aggregation_level'].isin(['daily', 'weekly'])) & (df_final['climate_cdd_at'].notna())].shape[0]} (should be 0)")

print("\n" + "="*70)

FINAL CONSOLIDATED DATASET DIAGNOSTICS

1. BASIC STRUCTURE:
   Shape: (4622, 19)
   Columns (19): ['date', 'year', 'month', 'quarter', 'week', 'aggregation_level', 'price_exaa_mean', 'price_mc_auction_mean', 'price_count_exaa', 'price_count_mc', 'data_completeness', 'month_name', 'carbonprices_exchange_rate_eur_usd', 'carbonprices_primary_market', 'carbonprices_secondary_market', 'climate_cdd_at', 'climate_hdd_at', 'climate_cdd_eu', 'climate_hdd_eu']

2. AGGREGATION LEVEL DISTRIBUTION:
aggregation_level
daily      3896
monthly     160
weekly      566
Name: count, dtype: int64

3. DATE RANGE BY AGGREGATION LEVEL:
   Daily: 2015-01-01 to 2025-08-31 (3896 rows)
   Weekly: 2015-01-05 to 2025-09-01 (566 rows)
   Monthly: 2012-05-01 to 2025-08-01 (160 rows)

4. MISSING VALUES BY DATA SOURCE:

   Electricity columns (4):
      price_exaa_mean: 1182 missing (25.6%)
      price_mc_auction_mean: 2183 missing (47.2%)
      price_count_exaa: 32 missing (0.7%)
      price_count_mc: 32 missing (0.7%

In [48]:
# Diagnostic Analysis with Absolute Path

import pandas as pd
from pathlib import Path

# Use absolute path
consolidated_path = Path(r"C:\Users\paulr\Desktop\Capstone\data\processed\data_consolidated.csv")

df_final = pd.read_csv(consolidated_path)

print("="*70)
print("AUTOMATED PIPELINE - CONSOLIDATED DATASET DIAGNOSTICS")
print("="*70)

# 1. Basic Structure
print("\n1. BASIC STRUCTURE:")
print(f"   Shape: {df_final.shape}")
print(f"   Columns ({len(df_final.columns)}):")
for i, col in enumerate(df_final.columns, 1):
    print(f"      {i}. {col}")

# 2. Aggregation Level Distribution
print("\n2. AGGREGATION LEVEL DISTRIBUTION:")
print(df_final['aggregation_level'].value_counts().sort_index())

# 3. Date Range by Aggregation Level
print("\n3. DATE RANGE BY AGGREGATION LEVEL:")
for level in ['daily', 'weekly', 'monthly']:
    level_data = df_final[df_final['aggregation_level'] == level]
    if len(level_data) > 0:
        print(f"   {level.capitalize()}: {level_data['date'].min()} to {level_data['date'].max()} ({len(level_data)} rows)")

# 4. Missing Values by Data Source
print("\n4. MISSING VALUES BY DATA SOURCE:")

electricity_cols = [col for col in df_final.columns if 'price_exaa' in col or 'price_mc' in col or 'price_count' in col]
carbon_cols = [col for col in df_final.columns if col.startswith('carbonprices_')]
climate_cols = [col for col in df_final.columns if col.startswith('climate_')]

print(f"\n   Electricity columns ({len(electricity_cols)}):")
for col in electricity_cols:
    missing = df_final[col].isnull().sum()
    print(f"      {col}: {missing} missing ({missing/len(df_final)*100:.1f}%)")

print(f"\n   Carbon columns ({len(carbon_cols)}):")
for col in carbon_cols:
    missing = df_final[col].isnull().sum()
    print(f"      {col}: {missing} missing ({missing/len(df_final)*100:.1f}%)")

print(f"\n   Climate columns ({len(climate_cols)}):")
for col in climate_cols:
    missing = df_final[col].isnull().sum()
    print(f"      {col}: {missing} missing ({missing/len(df_final)*100:.1f}%)")

# 5. Data Types
print("\n5. DATA TYPES VERIFICATION:")
type_checks = {
    'price_exaa_mean': 'Electricity prices',
    'carbonprices_primary_market': 'Carbon prices',
    'carbonprices_exchange_rate_eur_usd': 'Carbon exchange rate',
    'climate_cdd_at': 'Climate degree days'
}

for col, desc in type_checks.items():
    if col in df_final.columns:
        print(f"   {desc}: {df_final[col].dtype}")

# 6. Sample Monthly Data
print("\n6. SAMPLE MONTHLY DATA:")
monthly_sample = df_final[df_final['aggregation_level'] == 'monthly'].head(5)
display_cols = ['date', 'aggregation_level', 'price_exaa_mean', 'carbonprices_primary_market', 
                'climate_cdd_at', 'climate_hdd_at']
existing_cols = [col for col in display_cols if col in monthly_sample.columns]
print(monthly_sample[existing_cols].to_string(index=False))

# 7. Climate Architecture Check
print("\n7. CLIMATE ARCHITECTURE VERIFICATION:")
print(f"   Total rows: {len(df_final)}")
print(f"   Rows with climate data: {df_final['climate_cdd_at'].notna().sum()}")

daily_climate = len(df_final[(df_final['aggregation_level'] == 'daily') & (df_final['climate_cdd_at'].notna())])
weekly_climate = len(df_final[(df_final['aggregation_level'] == 'weekly') & (df_final['climate_cdd_at'].notna())])

print(f"   Daily with climate: {daily_climate} (should be 0)")
print(f"   Weekly with climate: {weekly_climate} (should be 0)")
print(f"   Climate monthly-only: {'CORRECT' if daily_climate == 0 and weekly_climate == 0 else 'ERROR'}")

# 8. Verification
print("\n8. EXPECTED VALUES:")
print(f"   Shape matches (4622, 19): {df_final.shape == (4622, 19)}")

print("\n" + "="*70)

AUTOMATED PIPELINE - CONSOLIDATED DATASET DIAGNOSTICS

1. BASIC STRUCTURE:
   Shape: (4622, 19)
   Columns (19):
      1. date
      2. year
      3. month
      4. quarter
      5. week
      6. aggregation_level
      7. price_exaa_mean
      8. price_mc_auction_mean
      9. price_count_exaa
      10. price_count_mc
      11. data_completeness
      12. month_name
      13. carbonprices_exchange_rate_eur_usd
      14. carbonprices_primary_market
      15. carbonprices_secondary_market
      16. climate_cdd_at
      17. climate_hdd_at
      18. climate_cdd_eu
      19. climate_hdd_eu

2. AGGREGATION LEVEL DISTRIBUTION:
aggregation_level
daily      3896
monthly     160
weekly      566
Name: count, dtype: int64

3. DATE RANGE BY AGGREGATION LEVEL:
   Daily: 2015-01-01 to 2025-08-31 (3896 rows)
   Weekly: 2015-01-05 to 2025-09-01 (566 rows)
   Monthly: 2012-05-01 to 2025-08-01 (160 rows)

4. MISSING VALUES BY DATA SOURCE:

   Electricity columns (4):
      price_exaa_mean: 1182 missing 